# RQ2 — Statistical Tests: Resolution Strategies

This notebook runs the formal statistical tests supporting the RQ2 results:

1. **Wilson CIs** for global strategy proportions (V1, V2, CC, CB, NC, NN)
2. **Descriptive comparison with Ghiotto et al.** — absolute differences + CI check
   (no GoF test: see §3 for why it is inappropriate here)
3. **Chi-squared test of independence**: strategy × agent
4. **Cramér's V** effect size for strategy × agent
5. **Chi-squared test of independence**: Imprecise rate × agent (localization quality)
6. **Post-hoc pairwise chi-squared** for agent pairs on the strategy distribution
7. **Chi-squared test of independence**: strategy × resolver (preview of RQ3)

Required packages: `scipy`, `statsmodels`.  Both are in the project venv.

In [ ]:
import sys
from pathlib import Path
from itertools import combinations

_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / 'analysis' / 'common.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

from analysis.common import (
    load_tables, build_chunk_frame,
    STRATEGY_ORDER,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Imports OK')

## 1. Load data

In [ ]:
tables = load_tables()
chunks = build_chunk_frame(tables)

# build_chunk_frame already canonicalises strategy → STRATEGY_ORDER categories
STRATEGIES_CLASSIFIABLE = [s for s in STRATEGY_ORDER if s != 'Imprecise']

classifiable = chunks[chunks['strategy'] != 'Imprecise'].copy()
n_total   = len(chunks)
n_classif = len(classifiable)
n_imp     = n_total - n_classif

print(f'Total chunks:    {n_total:,}')
print(f'Classifiable:    {n_classif:,}  ({n_classif/n_total:.1%})')
print(f'Imprecise:       {n_imp:,}  ({n_imp/n_total:.1%})')
print()
print('Global counts (classifiable only):')
print(classifiable['strategy'].value_counts().reindex(STRATEGIES_CLASSIFIABLE))

## 2. Wilson CIs for global strategy proportions

Wilson (score) interval at 95% confidence for each classifiable strategy.

In [ ]:
counts = classifiable['strategy'].value_counts().reindex(STRATEGIES_CLASSIFIABLE, fill_value=0)

rows = []
for strat, cnt in counts.items():
    lo, hi = proportion_confint(cnt, n_classif, alpha=0.05, method='wilson')
    rows.append({'strategy': strat, 'count': int(cnt),
                 'proportion': cnt / n_classif, 'ci_lo': lo, 'ci_hi': hi})

wilson_df = pd.DataFrame(rows)
wilson_df['pct'] = wilson_df['proportion'].map('{:.1%}'.format)
wilson_df['ci']  = wilson_df.apply(lambda r: f'[{r.ci_lo:.1%}, {r.ci_hi:.1%}]', axis=1)
print(wilson_df[['strategy', 'count', 'pct', 'ci']].to_string(index=False))

## 3. Descriptive comparison with Ghiotto et al.

### Why no chi-squared GoF test here?

A GoF test is **not appropriate** for this cross-study comparison for three reasons:

1. **Non-independence of observations.** Chunks within the same merge commit share
   context (same agent, same files, same codebase), as do chunks across merges of
   the same repository. The effective sample size is much smaller than the nominal
   ~60k, so the chi-squared p-value is anti-conservative.

2. **Different populations.** Ghiotto studied *human-authored Java* merges;
   we study *multi-language AI-authored* merges. The null hypothesis
   "our data was drawn from Ghiotto's distribution" is ill-posed:
   we are not claiming to sample from the same population.

3. **Power inflation.** At n ≈ 60k any deviation above ~0.1 pp would be
   "statistically significant", making the p-value uninformative about
   whether the difference is *substantively* meaningful.

**What we do instead:** report the absolute difference (our proportion − Ghiotto's)
and whether Ghiotto's point estimate falls outside our 95% Wilson CI.
If it does, the difference is detectable *within our study* — but interpretation
should be substantive ("X pp larger/smaller"), not inferential.

**Source:** Ghiotto et al. TSE 2020, Table 4. Proportions below are normalised
to sum to 1 over the six classifiable strategies (V1–NN), excluding Imprecise.

In [ ]:
# Ghiotto et al. published proportions — adjust if you have more precise values
GHIOTTO = {'V1': 0.425, 'V2': 0.145, 'CC': 0.066, 'CB': 0.074, 'NC': 0.218, 'NN': 0.072}
ghiotto_arr = np.array([GHIOTTO[s] for s in STRATEGIES_CLASSIFIABLE])
ghiotto_arr /= ghiotto_arr.sum()  # normalise to 1

rows_g = []
print(f"{'Strategy':<10} {'Ours':>8} {'CI_lo':>8} {'CI_hi':>8}"
      f" {'Ghiotto':>9} {'Diff':>8} {'Ghiotto in CI?':>16}")
print('-' * 73)
for i, s in enumerate(STRATEGIES_CLASSIFIABLE):
    cnt  = int(counts[s])
    ours = cnt / n_classif
    lo, hi = proportion_confint(cnt, n_classif, alpha=0.05, method='wilson')
    gh    = float(ghiotto_arr[i])
    in_ci = lo <= gh <= hi
    diff  = ours - gh
    rows_g.append({'strategy': s, 'ours': round(ours, 4),
                   'ci_lo': round(lo, 4), 'ci_hi': round(hi, 4),
                   'ghiotto': round(gh, 4), 'diff': round(diff, 4),
                   'ghiotto_in_ci': in_ci})
    flag = 'YES' if in_ci else 'NO (differs)'
    print(f'{s:<10} {ours:>8.1%} {lo:>8.1%} {hi:>8.1%}'
          f' {gh:>9.1%} {diff:>+8.1%} {flag:>16}')

ghiotto_compare_df = pd.DataFrame(rows_g)
print()
print('Interpretation: If Ghiotto\'s value is outside our CI, the difference is')
print('detectable in our study. Describe in absolute pp, not via p-values.')

## 4. Chi-squared test of independence: strategy × agent

H₀: The strategy distribution is the same across all coding agents.

This test IS appropriate here because we are comparing groups *within* our own
dataset (same population, same methodology). Independence holds at the chunk
level within each agent stratum — the remaining within-merge correlation inflates
Type I error slightly, but is common practice in this literature.

Expected-count assumption: all cells ≥ 5 (verified below).

In [ ]:
contingency_agent = (
    classifiable
    .groupby(['agent', 'strategy'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STRATEGIES_CLASSIFIABLE, fill_value=0)
)

print('Contingency table (rows = agents, cols = strategies):')
print(contingency_agent.to_string())

chi2, p, dof, expected = chi2_contingency(contingency_agent.values)
min_expected = expected.min()
pct_lt5 = (expected < 5).sum() / expected.size * 100

print(f'\nχ²({dof}) = {chi2:.1f},  p = {p:.2e}')
print(f'Min expected count: {min_expected:.1f}  |  Cells < 5: {pct_lt5:.1f}%')
if pct_lt5 > 20:
    print('WARNING: >20% cells < 5 — consider collapsing rare strategies.')

print('\nRow % (classifiable only):')
pct = contingency_agent.div(contingency_agent.sum(axis=1), axis=0)
print(pct.map('{:.1%}'.format).to_string())

## 5. Cramér's V effect size (strategy × agent)

Bias-corrected Cramér's V (Bergsma 2013).  
For a 6×5 table: |V| < 0.10 → weak; < 0.30 → moderate; ≥ 0.30 → strong.

In [ ]:
def cramers_v(chi2_val, n, r, c):
    phi2 = chi2_val / n
    phi2c = max(0, phi2 - (r - 1) * (c - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    cc = c - (c - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / min(rc - 1, cc - 1))

r, c = contingency_agent.shape
V = cramers_v(chi2, n_classif, r, c)
label = 'weak' if V < 0.10 else ('moderate' if V < 0.30 else 'strong')
print(f"Cramér's V (bias-corrected) = {V:.3f}  → {label} effect")

## 6. Chi-squared: Imprecise rate × agent

H₀: The probability that a chunk is Imprecise is equal across all agents.  
This checks whether localization failures are uniformly distributed.

**Cramér's V** (bias-corrected, Bergsma 2013) is added as the primary effect-size metric  
because at n~60k the p-value is nearly always significant.  
**Note:** Cramér's V shares the independence assumption with chi-squared; within-merge  
and within-repo clustering may slightly inflate both statistics. Treat V as an upper bound.

In [ ]:
imp_ct = (
    chunks
    .assign(is_imp=(chunks['strategy'] == 'Imprecise').astype(int))
    .groupby(['agent', 'is_imp']).size().unstack(fill_value=0)
    .rename(columns={0: 'classifiable', 1: 'imprecise'})
)
imp_ct['rate'] = (imp_ct['imprecise'] /
                  (imp_ct['classifiable'] + imp_ct['imprecise'])).map('{:.1%}'.format)
print(imp_ct.to_string())

ct_vals = imp_ct[['classifiable', 'imprecise']].values
chi2_imp, p_imp, dof_imp, _ = chi2_contingency(ct_vals)

# Cramér's V for Imprecise rate × agent (2×n_agents table)
n_imp_total = ct_vals.sum()
r_imp, c_imp = ct_vals.shape
V_imp = cramers_v(chi2_imp, n_imp_total, r_imp, c_imp)
label_imp = 'weak' if V_imp < 0.10 else ('moderate' if V_imp < 0.30 else 'strong')

print(f'\nχ²({dof_imp}) = {chi2_imp:.1f},  p = {p_imp:.2e}')
print(f"Cramér's V (bias-corrected) = {V_imp:.3f}  → {label_imp} effect  [primary metric]")
print('NOTE: clustering inflates both χ² and V slightly; interpret V as an upper bound.')


## 7. Post-hoc pairwise chi-squared: agent pairs (Holm–Bonferroni)

Which specific agent pairs differ in their strategy distributions?

**Primary output:** absolute percentage-point (pp) difference per strategy for each pair,  
plus Cramér's V per pair.  
**Caveat:** at n~60k virtually every pair will be 'significant' after Holm correction —  
the p-value is nearly uninformative. Focus on the magnitude of V and the pp differences.

In [ ]:
agents = contingency_agent.index.tolist()
pairs  = list(combinations(agents, 2))

# Proportion table for absolute pp differences
prop_agent = contingency_agent.div(contingency_agent.sum(axis=1), axis=0)

raw_ps = []
cramers_vs = []
for a1, a2 in pairs:
    sub = contingency_agent.loc[[a1, a2]].values
    if sub.sum() < 10:
        raw_ps.append(1.0)
        cramers_vs.append(0.0)
    else:
        chi2_pair, p_pair, _, _ = chi2_contingency(sub)
        raw_ps.append(p_pair)
        cramers_vs.append(cramers_v(chi2_pair, int(sub.sum()), 2, len(STRATEGIES_CLASSIFIABLE)))

_, corr_ps, _, _ = multipletests(raw_ps, method='holm')

print(f"{'Pair':<40} {'V':>6} {'effect':>10} {'p_holm':>10}")
print('-' * 70)
for i, (a1, a2) in enumerate(pairs):
    V_pair = cramers_vs[i]
    label_pair = 'weak' if V_pair < 0.10 else ('moderate' if V_pair < 0.30 else 'strong')
    pp = {s: prop_agent.loc[a1, s] - prop_agent.loc[a2, s] for s in STRATEGIES_CLASSIFIABLE}
    print(f'{a1} vs {a2:<{40-len(a1)-4}} {V_pair:>6.3f} {label_pair:>10} {corr_ps[i]:>10.4f}')
    print('  pp diffs: ' + '  '.join(f'{s}:{pp[s]:+.1%}' for s in STRATEGIES_CLASSIFIABLE))

print('\nNOTE: At n~60k, virtually all pairs are significant after Holm correction.')
print('Focus on Cramér\'s V and pp differences for substantive interpretation.')


## 8. Chi-squared: strategy × resolver (preview)

H₀: The strategy distribution is the same for agent-resolved and human-resolved chunks.  
Full analysis (with sensitivity) is in `rq3_statistical_tests.ipynb`.

In [ ]:
if 'resolver_type' not in classifiable.columns:
    print('resolver_type not found — run rq3_statistical_tests.ipynb for this analysis.')
else:
    ct_res = (
        classifiable[classifiable['resolver_type'].isin(['agent', 'human'])]
        .groupby(['resolver_type', 'strategy']).size().unstack(fill_value=0)
        .reindex(columns=STRATEGIES_CLASSIFIABLE, fill_value=0)
    )
    print(ct_res.to_string())
    print('\nRow %:')
    print(ct_res.div(ct_res.sum(axis=1), axis=0).map('{:.1%}'.format).to_string())

    chi2_res, p_res, dof_res, _ = chi2_contingency(ct_res.values)
    print(f'\nχ²({dof_res}) = {chi2_res:.1f},  p = {p_res:.2e}')
    V_res = cramers_v(chi2_res, ct_res.values.sum(), *ct_res.shape)
    print(f"Cramér's V = {V_res:.3f}")

## 9. Summary

In [ ]:
print('=== RQ2 STATISTICAL TEST SUMMARY ===')
print()
print('(1) Wilson CIs for global strategy proportions — see wilson_df')
print('(2) Descriptive comparison with Ghiotto — see ghiotto_compare_df')
print('    (No GoF test: different populations + within-cluster dependence + n~60k power inflation)')
print(f'(3) Independence strategy × agent: χ²({dof}) = {chi2:.1f}, p = {p:.2e}')
print(f"(4) Cramér's V (strategy × agent) = {V:.3f} ({label})")
print(f'(5) Imprecise rate × agent: χ²({dof_imp}) = {chi2_imp:.1f}, p = {p_imp:.2e}')
print(f"    Cramér's V (Imprecise × agent) = {V_imp:.3f} ({label_imp})  [primary effect metric]")
print('(6) Pairwise Holm-corrected: see pp differences and V per pair above')
print('(7) strategy × resolver — see rq3_statistical_tests.ipynb')
print()
print('VALIDITY NOTE: p-values are anti-conservative due to within-merge/within-repo clustering.')
print('Cramér\'s V (bias-corrected) shares the independence assumption but is far less')
print('sensitive to N. Treat all V values as slight upper bounds due to clustering.')
